# Notebook 11 – Advanced Cross Validation

## 1. K-Fold Cross Validation

### Topic Name

**K-Fold Cross Validation**

## Simple Concept

Instead of using only one train/validation split, K-Fold divides the dataset into **K parts (folds)**.

The model is trained and validated multiple times.

Example with `K = 5`:

* Round 1 → Fold 1 validation, remaining 4 training
* Round 2 → Fold 2 validation, remaining 4 training
* Round 3 → Fold 3 validation, remaining 4 training
* Round 4 → Fold 4 validation, remaining 4 training
* Round 5 → Fold 5 validation, remaining 4 training

Finally, we calculate the **average score**.

### When to use it

Use K-Fold when:

* Dataset is reasonably sized.
* Data is independent.
* Target is not heavily imbalanced.
* There is no time order or group dependency.

### When not to use it

Do not use normal K-Fold when:

* Classes are highly imbalanced → use **Stratified K-Fold**.
* Same customer/patient appears multiple times → use **Group K-Fold**.
* Data is time-dependent → use **Time Series Split**.

## Python Implementation

```python
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np

X = np.array([
    [1], [2], [3], [4], [5],
    [6], [7], [8], [9], [10]
])

y = np.array([
    10, 20, 30, 40, 50,
    60, 70, 80, 90, 100
])

model = LinearRegression()

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=kf,
    scoring="r2"
)

print("Fold Scores:", scores)
print("Average Score:", scores.mean())
```

### Output

```text
Fold Scores: [1. 1. 1. 1. 1.]
Average Score: 1.0
```

### What the Example Tells

The model was tested **5 different times** using different validation portions of the data.

The average score gives a more reliable estimate than using only one train/test split.

### Real-World Example

Predicting **house prices** where every house is an independent observation.

---

# 2. Stratified K-Fold

### Topic Name

**Stratified K-Fold Cross Validation**

## Simple Concept

Stratified K-Fold is mainly used for **classification**.

It makes sure that each fold has approximately the **same class distribution** as the complete dataset.

For example:

```text
Original dataset

Class 0 → 80%
Class 1 → 20%
```

Each fold will try to maintain:

```text
Class 0 → ~80%
Class 1 → ~20%
```

### Why do we need it?

Suppose we have:

```text
950 normal transactions
50 fraud transactions
```

A normal K-Fold split could create a validation fold with very few fraud cases.

Stratified K-Fold helps keep fraud cases represented across folds.

## Python Implementation

```python
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import numpy as np

X = np.array([
    [10], [20], [30], [40], [50],
    [60], [70], [80], [90], [100]
])

y = np.array([
    0, 0, 0, 0, 0,
    1, 1, 1, 1, 1
])

model = LogisticRegression()

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=skf,
    scoring="accuracy"
)

print("Fold Scores:", scores)
print("Average Accuracy:", scores.mean())
```

### What the Example Tells

Each fold tries to maintain the same proportion of:

```text
Class 0
Class 1
```

### Real-World Example

**Fraud detection**

```text
Normal → 95%
Fraud  → 5%
```

Stratified K-Fold makes sure every fold contains roughly the same fraud/normal ratio.

---

# 3. Repeated K-Fold

### Topic Name

**Repeated K-Fold Cross Validation**

## Simple Concept

Repeated K-Fold runs **K-Fold multiple times** with different splits.

For example:

```text
5 folds × 3 repeats = 15 model evaluations
```

Instead of depending on one particular K-Fold split, we repeat the process.

### When to use it

Useful when:

* Dataset is small or medium-sized.
* You want a more stable performance estimate.
* Results may change depending on the random split.

### When not to use it

Avoid it when:

* Dataset is extremely large and computation is expensive.
* Data has groups or time dependency.

## Python Implementation

```python
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np

X = np.arange(1, 21).reshape(-1, 1)

y = np.array([
    12, 19, 31, 39, 52,
    58, 71, 79, 91, 101,
    111, 121, 132, 139, 151,
    160, 171, 179, 191, 201
])

model = LinearRegression()

rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=rkf,
    scoring="r2"
)

print("Number of Scores:", len(scores))
print("Average Score:", scores.mean())
```

### Output

```text
Number of Scores: 15
Average Score: 0.99
```

The exact score can vary slightly depending on the data.

### What the Example Tells

With:

```python
n_splits=5
n_repeats=3
```

we get:

```text
5 × 3 = 15 evaluations
```

This gives us more observations of model performance.

---

# 4. Leave-One-Out Cross Validation

### Topic Name

**Leave-One-Out Cross Validation (LOOCV)**

## Simple Concept

Leave-One-Out means:

> Use **one row for validation** and all remaining rows for training.

If there are 10 rows:

```text
Round 1 → 9 train + 1 validation
Round 2 → 9 train + 1 validation
...
Round 10 → 9 train + 1 validation
```

So the model runs **10 times**.

## Python Implementation

```python
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np

X = np.array([
    [1], [2], [3], [4], [5]
])

y = np.array([
    10, 20, 30, 40, 50
])

model = LinearRegression()

loo = LeaveOneOut()

scores = cross_val_score(
    model,
    X,
    y,
    cv=loo,
    scoring="r2"
)

print("Number of Evaluations:", len(scores))
print("Scores:", scores)
```

### Output

```text
Number of Evaluations: 5
Scores: [...]
```

### When to use it

Useful when:

* Dataset is **very small**.
* You want to use almost every row for training in every round.

### When not to use it

Not recommended for large datasets because:

```text
1000 rows → 1000 model trainings
10000 rows → 10000 model trainings
```

This can become computationally expensive.

### Real-World Example

A research dataset containing only **30 patients**.

---

# 5. Group K-Fold

### Topic Name

**Group K-Fold Cross Validation**

## Simple Concept

Group K-Fold is used when multiple rows belong to the **same group**.

Important rule:

> Data from the same group should not appear in both training and validation.

### Example

Suppose we have hospital data:

```text
Patient A → 5 records
Patient B → 5 records
Patient C → 5 records
```

If Patient A's records appear in both training and validation, the model may indirectly see information about the same patient during training.

This can produce **data leakage**.

Group K-Fold keeps each patient's records together.

## Python Implementation

```python
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
import numpy as np

X = np.array([
    [10], [11], [12],
    [20], [21], [22],
    [30], [31], [32],
    [40], [41], [42]
])

y = np.array([
    0, 0, 1,
    0, 1, 1,
    0, 0, 1,
    1, 1, 1
])

groups = np.array([
    "Patient_A", "Patient_A", "Patient_A",
    "Patient_B", "Patient_B", "Patient_B",
    "Patient_C", "Patient_C", "Patient_C",
    "Patient_D", "Patient_D", "Patient_D"
])

model = RandomForestClassifier(
    n_estimators=50,
    random_state=42
)

gkf = GroupKFold(n_splits=4)

scores = cross_val_score(
    model,
    X,
    y,
    groups=groups,
    cv=gkf,
    scoring="accuracy"
)

print("Fold Scores:", scores)
print("Average Accuracy:", scores.mean())
```

### What the Example Tells

The model will never use records from the same patient in both:

```text
Training
   ↓
Validation
```

The entire patient group stays together.

### Real-World Examples

Group K-Fold is useful for:

* Multiple medical records from the same patient
* Multiple purchases from the same customer
* Multiple images from the same person
* Multiple measurements from the same machine
* Multiple transactions from the same account

---

# 6. Time Series Split

### Topic Name

**Time Series Cross Validation**

## Simple Concept

Time Series data has an important rule:

> **Past should predict future.**

We should not randomly mix future data into training.

For example:

```text
January → February → March → April → May
```

Training should happen like:

```text
Train: January
Test:  February

Train: January + February
Test:  March

Train: January + February + March
Test:  April

Train: January + February + March + April
Test:  May
```

## Python Implementation

```python
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

X = np.arange(1, 11)

tscv = TimeSeriesSplit(n_splits=4)

for train_index, test_index in tscv.split(X):

    print("Train:", X[train_index])
    print("Test :", X[test_index])
```

### Output

```text
Train: [1 2]
Test : [3 4]

Train: [1 2 3 4]
Test : [5 6]

Train: [1 2 3 4 5 6]
Test : [7 8]

Train: [1 2 3 4 5 6 7 8]
Test : [9 10]
```

### What the Example Tells

The training data always comes **before** the validation data.

The model never gets to train on future observations before predicting them.

### Real-World Examples

Use Time Series Split for:

* Stock/financial data
* Daily sales
* Monthly revenue
* Website traffic
* Electricity demand
* Weather measurements
* Sensor readings

---

# Choosing the Correct Validation Strategy

| Dataset Type                             | Recommended Strategy |
| ---------------------------------------- | -------------------- |
| Normal independent data                  | K-Fold               |
| Classification with class imbalance      | Stratified K-Fold    |
| Small dataset needing stable estimate    | Repeated K-Fold      |
| Very small dataset                       | Leave-One-Out        |
| Multiple rows from same customer/patient | Group K-Fold         |
| Time-dependent data                      | Time Series Split    |

## Easy Decision Rule

Think about the dataset first.

### Question 1: Is the data time-dependent?

```text
YES → Time Series Split
NO  → Continue
```

### Question 2: Do multiple rows belong to the same person/customer/machine?

```text
YES → Group K-Fold
NO  → Continue
```

### Question 3: Is it classification with an imbalanced target?

```text
YES → Stratified K-Fold
NO  → Continue
```

### Question 4: Is the dataset very small?

```text
YES → Leave-One-Out or Repeated K-Fold
NO  → K-Fold
```

# Key Points

* **K-Fold** → general-purpose validation.
* **Stratified K-Fold** → classification + maintain class proportions.
* **Repeated K-Fold** → repeat K-Fold to get a more stable estimate.
* **Leave-One-Out** → one row is validation at a time; useful for very small datasets.
* **Group K-Fold** → keeps related records together and helps prevent group leakage.
* **Time Series Split** → respects chronological order.
* The correct validation strategy depends on the **structure of your dataset**, not just the model you are using.
